In [1]:
#pip install cmake gym[atari] scipy

# fuerza bruta vs q-learning

In [1]:
#import gym
import gymnasium as gym

In [10]:
# Creation of the environment
env = gym.make("Taxi-v3", render_mode="human")

#env = gym.make("Taxi-v3").env
#env.render()

In [11]:
env.reset() # reset the environment to a new random state
env.render() 

print("Action Space {}". format(env.action_space))
print("State Space {}.". format(env.observation_space))

Action Space Discrete(6)
State Space Discrete(500).


In [12]:
print(env)
env = env.unwrapped # for gymnasium
print(env)
state = env.encode(3, 1, 2, 0) # (taxi row, taxi column, passenger index, destination index)
print("State:", state)

env.s = state
env.render()


<TimeLimit<OrderEnforcing<PassiveEnvChecker<TaxiEnv<Taxi-v3>>>>>
<TaxiEnv<Taxi-v3>>
State: 328


In [13]:
env.P[328]

{0: [(1.0, 428, -1, False)],
 1: [(1.0, 228, -1, False)],
 2: [(1.0, 348, -1, False)],
 3: [(1.0, 328, -1, False)],
 4: [(1.0, 328, -10, False)],
 5: [(1.0, 328, -10, False)]}

In [14]:
env.close()

In [16]:

env = gym.make("Taxi-v3", render_mode="rgb_array")
env.reset()

# without reinforcement learning
env.s = 328  # set environment to illustration's state

epochs = 0
penalties, reward = 0, 0

frames = [] # for animation

done = False

while not done:
    action = env.action_space.sample()
    state, reward, done, truncate, info = env.step(action)

    if reward == -10:
        penalties += 1
    
    # Put each rendered frame into dict for animation
    frames.append({
        'frame': env.render(),
        'state': state,
        'action': action,
        'reward': reward
        }
    )

    epochs += 1
    
    
print("Timesteps taken: {}".format(epochs))
print("Penalties incurred: {}".format(penalties))

Timesteps taken: 731
Penalties incurred: 229


In [17]:
from IPython.display import clear_output
from time import sleep

def print_frames(frames):
    for i, frame in enumerate(frames):
        clear_output(wait=True)
        #print(frame['frame'].getvalue())
        print(f"{frame['frame']}")
        print(f"Timestep: {i + 1}")
        print(f"State: {frame['state']}")
        print(f"Action: {frame['action']}")
        print(f"Reward: {frame['reward']}")
        sleep(.1)
        
print_frames(frames)

[[[110 109 106]
  [110 109 106]
  [124 122 122]
  ...
  [108 111 109]
  [108 111 109]
  [118 119 119]]

 [[110 109 106]
  [110 109 106]
  [124 122 122]
  ...
  [108 111 109]
  [108 111 109]
  [118 119 119]]

 [[114 116 115]
  [114 116 115]
  [126 127 126]
  ...
  [112 113 111]
  [112 113 111]
  [118 117 115]]

 ...

 [[116 115 116]
  [116 115 116]
  [106 107 108]
  ...
  [113 115 114]
  [113 115 114]
  [117 114 117]]

 [[116 115 116]
  [116 115 116]
  [106 107 108]
  ...
  [113 115 114]
  [113 115 114]
  [117 114 117]]

 [[115 112 112]
  [115 112 112]
  [119 119 117]
  ...
  [123 119 118]
  [123 119 118]
  [114 114 117]]]
Timestep: 731
State: 0
Action: 5
Reward: 20


In [18]:
import numpy as np
q_table = np.zeros([env.observation_space.n, env.action_space.n])

In [19]:
%%time
"""Training the agent"""

import random
from IPython.display import clear_output

# Hyperparameters
alpha = 0.1
gamma = 0.6
epsilon = 0.1

# For plotting metrics
all_epochs = []
all_penalties = []

for i in range(1, 100001):
    state = env.reset()
    state= state[0]

    epochs, penalties, reward, = 0, 0, 0
    done = False


    while not done:
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Explore action space
        else:
            action = np.argmax(q_table[state]) # Exploit learned values
        

        next_state, reward, done, truncate, info = env.step(action) 
  
        old_value = q_table[state, action]
        next_max = np.max(q_table[next_state])
  
        new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
        #print(state[0], action, new_value)
        q_table[state,action] = new_value
        #print(state[0], action, new_value)
        #print(reward)
        
        if reward == -10:
            penalties += 1

        state= next_state
        epochs += 1
        #print(state)
    if i % 100 == 0:
        clear_output(wait=True)
        print(f"Episode: {i}")

print("Training finished.\n")

Episode: 100000
Training finished.

CPU times: user 14 s, sys: 3.21 s, total: 17.2 s
Wall time: 16 s


In [12]:
q_table

array([[  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [ -2.41837061,  -2.3639511 ,  -2.41837066,  -2.36395109,
         -2.27325184, -11.36395014],
       [ -1.87014399,  -1.45024   ,  -1.870144  ,  -1.45024008,
         -0.7504    , -10.45023996],
       ...,
       [ -1.09322883,   0.41599931,  -1.0928844 ,  -1.29317558,
         -4.12033949,  -4.16980842],
       [ -2.17017972,  -2.12206133,  -2.17093962,  -2.1220621 ,
         -7.60496261,  -4.40748727],
       [  2.81689549,   0.85070691,   3.10501198,  11.        ,
         -1.82668699,  -2.2548507 ]], shape=(500, 6))

In [19]:
#env.close()

In [20]:
#env = gym.make("Taxi-v3", render_mode="ansi")
#env.reset()

In [13]:

"""Evaluate agent's performance after Q-learning"""

total_epochs, total_penalties = 0, 0
episodes = 100

framesql = [] # for animation

for _ in range(episodes):
    state = env.reset()
    state= state[0]
    epochs, penalties, reward = 0, 0, 0
    
    done = False
    
    while not done:
        action = np.argmax(q_table[state])
        state, reward, done, truncate, info = env.step(action)

        if reward == -10:
            penalties += 1
        
        epochs += 1

    total_penalties += penalties
    total_epochs += epochs
    # Put each rendered frame into dict for animation
    framesql.append({
    'frame': env.render(),
    'state': state,
    'action': action,
    'reward': reward,
    'epoch': epochs   
        }
    )

print(f"Results after {episodes} episodes:")
print(f"Average timesteps per episode: {total_epochs / episodes}")
print(f"Average penalties per episode: {total_penalties / episodes}")



Results after 100 episodes:
Average timesteps per episode: 13.07
Average penalties per episode: 0.0


In [14]:
env.close()
env = gym.make("Taxi-v3", render_mode="human")


In [15]:
# After 10 000 episodes, our Q-table can be used as a "cheatsheet" to play FrozenLake"
# Here the agent plays FrozenLake

env.reset()

max_steps = 100

for episode in range(10):
    state = env.reset()
    state=state[0]
    step = 0
    done = False
    print("****************************************************")
    print("EPISODE ", episode)

    for step in range(max_steps):
        
        # Take the action (index) that have the maximum expected future reward given that state
        action = np.argmax(q_table[state,:])
        
        new_state, reward, done, truncate, info = env.step(action)

        env.render()
        
        if done:
            # Here, we decide to only print the last state (to see if our agent is on the goal or fall into an hole)
            env.render()
            
            # We print the number of step it took.
            print("Number of steps", step)
            break
        state = new_state
env.close()

****************************************************
EPISODE  0
Number of steps 16
****************************************************
EPISODE  1
Number of steps 12
****************************************************
EPISODE  2
Number of steps 17
****************************************************
EPISODE  3
Number of steps 14
****************************************************
EPISODE  4
Number of steps 10
****************************************************
EPISODE  5
Number of steps 13
****************************************************
EPISODE  6
Number of steps 8
****************************************************
EPISODE  7
Number of steps 12
****************************************************
EPISODE  8
Number of steps 12
****************************************************
EPISODE  9
Number of steps 15
